In [ ]:
!pip install gradio
!pip install --upgrade accelerate transformers peft
!pip install --upgrade bitsandbytes
!pip install flash-attn --no-build-isolation

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_path)
base_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype="bfloat16",
 #   attn_implementation="flash_attention_2",
).to("cuda")

In [ ]:
model = PeftModel.from_pretrained(base_model, "maomao88/anime-waifu-mistral-lora").to("cuda")

In [ ]:
import torch

def chat_with_personality(trait, user_input, max_new_tokens=100, temperature=0.8):
    """
    Generate a response from the fine-tuned model using a given personality trait.
    """
    messages = [
        {
            "role": "system",
            "content": f"You are an anime character with the following personality: {trait}."  # 设置系统提示词，定义角色性格
        },
        {
            "role": "user",
            "content": user_input  # 用户输入的消息
        }
    ]

    # 应用 Mistral 的聊天模板，将消息格式化为模型可接受的提示词
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,  # 不进行分词，返回字符串
        add_generation_prompt=True  # 添加生成提示符
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)  # 将提示词编码为张量并移至模型所在设备

    with torch.no_grad():  # 关闭梯度计算，节省显存
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,  # 最大生成token数
            do_sample=True,  # 启用采样生成
            temperature=temperature,  # 温度参数，控制生成随机性
            top_p=0.9,  # 核采样，保留累计概率前90%的token
            pad_token_id=tokenizer.eos_token_id,  # 填充token设为结束符
            repetition_penalty=1.1  # 重复惩罚，减少重复输出
        )

    # 解码生成的token序列为文本，跳过特殊token
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # 移除提示词部分，只保留助手的回复
    if "[/INST]" in response:
        response = response.split("[/INST]")[-1].strip()

    return response

In [ ]:
tsundere = chat_with_personality("tsundere", "Do you like flowers?")
print(tsundere)

In [ ]:
trait = "tsundere"
MAX_HISTORY_ROUNDS = 6

In [ ]:
# 初始化对话，设置系统提示词
history = [
    {
        "role": "system",
        "content": f"You are an anime character with the following personality: {trait}."  # 定义角色性格的系统提示词
    }
]

print(f"--- Chat started with {trait} character! (Type 'quit' to stop) ---")  # 打印聊天开始提示

while True:
    user_input = input("You: ")  # 获取用户输入
    if user_input.lower() in ["quit", "exit"]:  # 输入 quit 或 exit 退出循环
        break

    # 将用户输入追加到对话历史
    history.append({"role": "user", "content": user_input})

    # 如果对话历史过长，裁剪保留最近的几轮，同时保留系统提示词
    if len(history) > MAX_HISTORY_ROUNDS * 2:
        history = [history[0]] + history[-(MAX_HISTORY_ROUNDS * 2 - 1):]

    # 应用聊天模板，将对话历史格式化为模型输入
    prompt = tokenizer.apply_chat_template(
        history,
        tokenize=False,  # 返回字符串而非 token ID
        add_generation_prompt=True  # 添加助手回复起始标记
    )

    # 将字符串编码为张量，并移至模型所在设备（GPU）
    inputs = tokenizer(
    prompt,
    return_tensors="pt"
    ).to(model.device)

    # 模型生成回复
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,  # 最大生成256个token
        temperature=0.7,  # 温度参数，控制随机性
        do_sample=True,  # 启用采样生成
        pad_token_id=tokenizer.eos_token_id  # 填充token设为结束符
    )

    # 解码输出，跳过提示词部分，只提取模型生成的回复
    response_ids = outputs[0][len(inputs['input_ids'][0]):]  # 截取生成部分的 token ID
    response = tokenizer.decode(response_ids, skip_special_tokens=True)  # 解码为文本

    print(f"Character: {response}")  # 打印角色回复

    # 将助手回复追加到对话历史，用于多轮对话
    history.append({"role": "assistant", "content": response})

In [ ]:
import gradio as gr

MAX_HISTORY_ROUNDS = 10

def predict(message, history):
    trait = "tsundere"
    messages = [
        {"role": "system", "content": f"You are an anime character with the following personality: {trait}."}
    ]

    # 添加之前的对话轮次（history 格式为 [[user_msg, assistant_msg], ...]）
    for user_msg, assistant_msg in history[-MAX_HISTORY_ROUNDS:]:
        messages.append({"role": "user", "content": user_msg})
        if assistant_msg:
            messages.append({"role": "assistant", "content": assistant_msg})

    # 添加当前用户消息
    messages.append({"role": "user", "content": message})

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.8,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1
        )

    response_ids = outputs[0][len(inputs['input_ids'][0]):]
    response = tokenizer.decode(response_ids, skip_special_tokens=True)

    return response

In [ ]:
demo = gr.ChatInterface(
    fn=predict,
    title="Anime Character Chat",
    description="Talk to a fine-tuned Mistral model",
    examples=[["Hello! Who are you?"], ["What is your favorite food?"], ["Nice to meet you."]],
)

In [ ]:
# theme 参数在创建 ChatInterface 时设置，不能放在 launch() 里
demo = gr.ChatInterface(fn=predict, theme="soft")
demo.launch(debug=True, server_port=7861)  # 启动服务，端口7861，开启调试模式